# 🏥 Dermatology AI Fine-Tuning (V2.1)

**EfficientNet-B4 (PyTorch)** fine-tuning from V2 checkpoint:
- ✅ MixUp Augmentation (Combats overfitting)
- ✅ Micro Learning Rates (For delicate boundary refinement)
- ✅ Weights initialized from `efficientnet_b4_derma_v2.pth`
- ✅ All layers unfrozen directly

**Target**: >80% validation accuracy

In [11]:
import os
import json
import glob
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B4_Weights
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

Device: cuda


## 1) Locate and extract dataset

In [ ]:
import os
import glob
import zipfile
from pathlib import Path

# --- Dataset and Model Setup ---
print("🔍 Available datasets in /kaggle/input:")
try:
    for item in os.listdir("/kaggle/input"):
        item_path = os.path.join("/kaggle/input", item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}")
            try:
                subdirs = os.listdir(item_path)[:5]
                for subdir in subdirs:
                    print(f"    └── {subdir}")
            except: pass
except Exception as e: print(e)

EXTRACT_DIR = None
PRETRAINED_MODEL = None

# Locate dataset
known_paths = [
    "/kaggle/input/derma-db-24class/processed",
    "/kaggle/input/dermadb-24class-v2/processed",
    "/kaggle/input/datasets/dnghongkhang/derma-db-24class/processed",
]
for path in known_paths:
    if Path(path, "train").exists():
        EXTRACT_DIR = path
        break

# Locate pretrained model (assuming user uploads it)
model_paths = glob.glob("/kaggle/input/**/*.pth", recursive=True)
for mp in model_paths:
    if "efficientnet_b4" in mp:
        PRETRAINED_MODEL = mp
        break

if not EXTRACT_DIR and Path("/kaggle/input").exists():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train" in dirs and "val" in dirs:
            EXTRACT_DIR = root
            break

print(f"\n✅ DATASET PATH: {EXTRACT_DIR}")
if PRETRAINED_MODEL:
    print(f"✅ PRETRAINED MODEL: {PRETRAINED_MODEL}")
else:
    print(f"⚠️ PRETRAINED MODEL NOT FOUND! (Will train from scratch if unavailable)")


## 2) Build train/val datasets with ENHANCED augmentation

In [ ]:
def unwrap_single_dir(path: str) -> str:
    p = Path(path)
    subdirs = [d for d in p.iterdir() if d.is_dir()]
    files = [f for f in p.iterdir() if f.is_file()]
    if len(subdirs) == 1 and not files:
        return str(subdirs[0])
    return str(p)

BASE_DIR = unwrap_single_dir(EXTRACT_DIR)
print("BASE_DIR:", BASE_DIR)

train_dir = None
val_dir = None

if Path(BASE_DIR, "train").exists():
    train_dir = str(Path(BASE_DIR, "train"))
if Path(BASE_DIR, "val").exists():
    val_dir = str(Path(BASE_DIR, "val"))
elif Path(BASE_DIR, "valid").exists():
    val_dir = str(Path(BASE_DIR, "valid"))
elif Path(BASE_DIR, "validation").exists():
    val_dir = str(Path(BASE_DIR, "validation"))

# EfficientNet-B4 config
weights = EfficientNet_B4_Weights.DEFAULT
IMG_SIZE = 380

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# 🔥 ENHANCED AUGMENTATION (Phase 3)
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),  # Increased from 15
    
    # NEW: Color augmentation for lighting variations
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    
    # NEW: Gaussian blur for focus variations
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
    
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
    
    # Increased erasing probability
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 20),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

VAL_SPLIT = 0.15

# Group-based split helper functions
def find_metadata_csv(base_dir: str):
    csvs = list(Path(base_dir).rglob("*.csv"))
    if not csvs:
        return None

    preferred = [
        "metadata.csv",
        "train_metadata.csv",
        "labels.csv",
        "isic_2019_training_groundtruth.csv",
    ]
    for name in preferred:
        for p in csvs:
            if p.name.lower() == name:
                return p

    for p in csvs:
        try:
            df_head = pd.read_csv(p, nrows=5)
        except Exception:
            continue
        cols = {c.lower() for c in df_head.columns}
        has_group = bool({"patient_id", "lesion_id", "patient", "lesion"} & cols)
        has_img = bool({"image", "image_id", "isic_id", "filename", "file_name", "image_name"} & cols)
        if has_group and has_img:
            return p
    return None

def build_group_map(df: pd.DataFrame):
    cols = {c.lower(): c for c in df.columns}
    group_col = None
    for key in ["patient_id", "patient", "lesion_id", "lesion"]:
        if key in cols:
            group_col = cols[key]
            break

    img_col = None
    for key in ["image", "image_id", "isic_id", "filename", "file_name", "image_name"]:
        if key in cols:
            img_col = cols[key]
            break

    if not group_col or not img_col:
        return None, None, None

    mapping = {}
    for img, grp in zip(df[img_col], df[group_col]):
        if pd.isna(img) or pd.isna(grp):
            continue
        key = str(img)
        mapping[key] = grp
        mapping[Path(key).name] = grp
        mapping[Path(key).stem] = grp

    return mapping, img_col, group_col

if train_dir and val_dir:
    train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
    val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)
    class_names = train_ds.classes

    metadata_csv = find_metadata_csv(BASE_DIR)
    if metadata_csv:
        print("Found metadata:", metadata_csv)
        df = pd.read_csv(metadata_csv)
        group_map, img_col, group_col = build_group_map(df)
        if group_map:
            def groups_from_samples(samples):
                groups = set()
                missing = 0
                for path, _ in samples:
                    p = Path(path)
                    g = group_map.get(p.stem) or group_map.get(p.name)
                    if g is None:
                        missing += 1
                        continue
                    groups.add(g)
                return groups, missing

            train_groups, miss_tr = groups_from_samples(train_ds.samples)
            val_groups, miss_val = groups_from_samples(val_ds.samples)
            overlap = train_groups & val_groups
            if overlap:
                print(
                    f"[WARNING] Potential leakage: {len(overlap)} groups in both train and val."
                )
            if miss_tr + miss_val > 0:
                print(
                    f"[WARNING] Missing group for {miss_tr + miss_val} images."
                )
else:
    full_train = datasets.ImageFolder(BASE_DIR, transform=train_tfms)
    full_val = datasets.ImageFolder(BASE_DIR, transform=val_tfms)
    class_names = full_train.classes

    labels = [y for _, y in full_train.samples]

    metadata_csv = find_metadata_csv(BASE_DIR)
    use_groups = False

    if metadata_csv:
        print("Found metadata:", metadata_csv)
        df = pd.read_csv(metadata_csv)
        group_map, img_col, group_col = build_group_map(df)
        if group_map:
            groups = []
            missing = 0
            for path, _ in full_train.samples:
                p = Path(path)
                g = group_map.get(p.stem) or group_map.get(p.name)
                if g is None:
                    missing += 1
                    g = f"unknown_{len(groups)}"
                groups.append(g)

            if missing > 0:
                print(f"[WARNING] Missing group for {missing} images.")

            gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=SEED)
            train_idx, val_idx = next(gss.split(np.zeros(len(labels)), labels, groups))
            use_groups = True
            print(f"Group split using column '{group_col}'.")
        else:
            print("[WARNING] Metadata found but columns not detected. Using stratified split.")

    if not use_groups:
        idxs_by_class = {}
        for i, y in enumerate(labels):
            idxs_by_class.setdefault(y, []).append(i)

        train_idx, val_idx = [], []
        rng = random.Random(SEED)
        for y, idxs in idxs_by_class.items():
            rng.shuffle(idxs)
            n_val = max(1, int(len(idxs) * VAL_SPLIT))
            val_idx.extend(idxs[:n_val])
            train_idx.extend(idxs[n_val:])

    train_ds = Subset(full_train, train_idx)
    val_ds = Subset(full_val, val_idx)

NUM_CLASSES = len(class_names)
print("Classes:", NUM_CLASSES)
if NUM_CLASSES != 24:
    print("[WARNING] Expected 24 classes, found:", NUM_CLASSES)

## 3) DataLoaders with WEIGHTED SAMPLING (Phase 1)

In [ ]:
# Extract labels from dataset
if isinstance(train_ds, Subset):
    base_ds = train_ds.dataset
    idxs = train_ds.indices
    labels = [base_ds.samples[i][1] for i in idxs]
else:
    labels = [y for _, y in train_ds.samples]

# 🔥 IMPROVED: Square root weighting to avoid extreme values
counts = np.bincount(labels, minlength=NUM_CLASSES)
class_weights = 1. / np.sqrt(counts + 1e-6)  # Square root instead of linear
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print("✅ Class Weights (sqrt-based) calculated")
print(f"   Min weight: {class_weights.min():.3f}, Max weight: {class_weights.max():.3f}")

# 🔥 CRITICAL FIX: Weighted Random Sampler for balanced batches
sample_weights = [class_weights[label].item() for label in labels]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("✅ WeightedRandomSampler created")

# Optimized hyperparameters (Phase 4)
BATCH_SIZE = 32  # Increased from 16
NUM_WORKERS = 4

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,  # Use sampler instead of shuffle
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2
)

print(f"Train batches: {len(train_loader)} (batch_size={BATCH_SIZE})")
print(f"Val batches: {len(val_loader)}")

## 4) Model (EfficientNet-B4)

In [ ]:
model = models.efficientnet_b4(weights=weights)

# Replace classifier for 24 classes
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.6, inplace=True),
    nn.Linear(in_features, NUM_CLASSES)
)

# Load V2 Weights if available
if PRETRAINED_MODEL:
    print(f"🔥 Loading pre-trained V2 weights from {PRETRAINED_MODEL}")
    ckpt = torch.load(PRETRAINED_MODEL, map_location=device, weights_only=False)
    if "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"])
    else:
        model.load_state_dict(ckpt)
    
# Unfreeze all layers immediately for Fine-Tuning
for param in model.parameters():
    param.requires_grad = True

model = model.to(device)
MODEL_TAG = "efficientnet_b4_derma_v2_1_finetuned.pth"
print("✅ Model initialized (All layers unfrozen for Fine-Tuning)")


## 5) Loss, optimizer, scheduler with DISCRIMINATIVE LR (Phase 2)

In [ ]:
# Loss with improved label smoothing
criterion = nn.CrossEntropyLoss(
    weight=class_weights, 
    label_smoothing=0.1 
)

# 🔥 MICRO LEARNING RATES (Fine-tuning V2)
LR_CLASSIFIER = 5e-5  # Was 1e-3
LR_BACKBONE = 1e-5    # Was 1e-4
WEIGHT_DECAY = 3e-2   # Keep strong

optimizer = torch.optim.AdamW([
    {'params': model.features.parameters(), 'lr': LR_BACKBONE},
    {'params': model.classifier.parameters(), 'lr': LR_CLASSIFIER}
], weight_decay=WEIGHT_DECAY)

NUM_EPOCHS = 30  # Less epochs needed for fine-tuning
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=15,  # Longer cycle
    T_mult=1,
    eta_min=1e-7
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

print(f"✅ Optimizer: AdamW (LR_classifier={LR_CLASSIFIER}, LR_backbone={LR_BACKBONE})")

# Test-Time Augmentation
def tta_forward(model, images):
    logits = model(images)
    logits += model(torch.flip(images, dims=[3]))
    logits += model(torch.flip(images, dims=[2]))
    logits += model(torch.rot90(images, 1, [2,3]))
    logits += model(torch.rot90(images, 3, [2,3]))
    return logits / 5


## 6) Train & validate with PROGRESSIVE UNFREEZING (Phase 2.1)

In [ ]:
import copy
from sklearn.metrics import f1_score

PATIENCE = 8  # Early stopping patience

# 🔥 OPTIMAL MIXUP HELPERS
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def get_class_to_idx(dataset):
    if hasattr(dataset, 'class_to_idx'): return dataset.class_to_idx
    if hasattr(dataset, 'dataset') and hasattr(dataset.dataset, 'class_to_idx'): return dataset.dataset.class_to_idx
    return None

def run_epoch(model, loader, train=True):
    all_preds = []
    all_labels = []
    
    if train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)
            
            # 🔥 Apply MixUp (50% chance)
            use_mixup = np.random.rand() > 0.5
            if use_mixup:
                images, targets_a, targets_b, lam = mixup_data(images, labels)
            
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(images)
                if use_mixup:
                    loss = mixup_criterion(criterion, logits, targets_a, targets_b, lam)
                else:
                    loss = criterion(logits, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5) 
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                logits = tta_forward(model, images)
                loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        
        # Don't record predictions effectively during mixup training for accuracy metric (it gets biased)
        # We only really care about val accuracy
        if not train or not use_mixup:
            preds = logits.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            if not train or not use_mixup: 
               # Careful tracking labels if mixup is off
               all_labels.extend(labels.cpu().numpy())

    # Fallback if all train batches were mixup (unlikely but possible for tiny sets)
    if train and len(all_labels) == 0:
        return running_loss / len(loader.dataset), 0.0, 0.0

    epoch_loss = running_loss / (len(all_labels) if len(all_labels) > 0 else len(loader.dataset))
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    epoch_acc = (all_preds == all_labels).mean() if len(all_labels)>0 else 0.0
    epoch_f1 = f1_score(all_labels, all_preds, average="macro") if len(all_labels)>0 else 0.0
    
    return epoch_loss, epoch_acc, epoch_f1

print("🚀 Starting FINE-TUNING with MixUp...")
best_val_f1 = 0.0
best_val_loss = float('inf')
val_loss_no_improve = 0
epochs_no_improve = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_f1 = run_epoch(model, val_loader, train=False)
    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss, "train_acc": train_acc, "train_f1": train_f1,
        "val_loss": val_loss, "val_acc": val_acc, "val_f1": val_f1,
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        val_loss_no_improve = 0
    else:
        val_loss_no_improve += 1
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save({
            "model_state": model.state_dict(),
            "class_to_idx": get_class_to_idx(train_ds),
            "val_f1": best_val_f1, "val_acc": val_acc
        }, MODEL_TAG)
        print(f"✅ Epoch {epoch} | SAVED BEST: F1={val_f1:.4f} (Acc={val_acc:.4f})")
    else:
        epochs_no_improve += 1
        train_val_gap = train_acc - val_acc
        if train_val_gap > 0.05:
            print(f"⚠️ Gap: {train_val_gap:.1%}")
    
    torch.save(model.state_dict(), "last_checkpoint.pth")
    print(f"   Train(NoMixup): Acc={train_acc:.4f} | Val: Acc={val_acc:.4f} F1={val_f1:.4f}")

    if epochs_no_improve >= PATIENCE:
        print(f"⏹️  Early Stopping at Epoch {epoch}.")
        break

print(f"🎉 Finished! Best F1: {best_val_f1:.4f}")


## 7) Save history & class mapping

In [ ]:
import json
import matplotlib.pyplot as plt

with open("training_history.json", "w") as f:
    json.dump(history, f, indent=2)

class_to_idx = None
if hasattr(train_ds, 'class_to_idx'):
    class_to_idx = train_ds.class_to_idx
elif hasattr(train_ds, 'dataset') and hasattr(train_ds.dataset, 'class_to_idx'):
    class_to_idx = train_ds.dataset.class_to_idx

if class_to_idx:
    with open("class_to_idx.json", "w") as f:
        json.dump(class_to_idx, f, indent=2)

if len(history) > 0:
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    train_acc = [h["train_acc"] for h in history]
    val_acc = [h["val_acc"] for h in history]
    train_f1 = [h["train_f1"] for h in history]
    val_f1 = [h["val_f1"] for h in history]

    plt.figure(figsize=(18, 4))
    
    plt.subplot(1, 3, 1)
    plt.plot(epochs, train_acc, label="Train Acc")
    plt.plot(epochs, val_acc, label="Val Acc")
    plt.legend()
    plt.title("Accuracy")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 2)
    plt.plot(epochs, train_f1, label="Train F1")
    plt.plot(epochs, val_f1, label="Val F1")
    plt.legend()
    plt.title("F1 Score (Macro)")
    plt.xlabel("Epoch")

    plt.subplot(1, 3, 3)
    plt.plot(epochs, train_loss, label="Train Loss")
    plt.plot(epochs, val_loss, label="Val Loss")
    plt.legend()
    plt.title("Loss")
    plt.xlabel("Epoch")

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=200)
    plt.show()
else:
    print("No history to plot.")

## 8) Evaluation (confusion matrix + report)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

ckpt = torch.load(MODEL_TAG, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

all_preds = []
all_labels = []

print("Evaluating on validation set with TTA...")
with torch.no_grad():
    for images, labels in tqdm(val_loader):
        images = images.to(device, non_blocking=True)
        logits = tta_forward(model, images)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

macro_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"\n🎯 MACRO F1 SCORE: {macro_f1:.4f}")
print("="*50)

print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=90, ha="center")
plt.title(f"Confusion Matrix (F1: {macro_f1:.4f})")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()

NameError: name 'torch' is not defined

## 9) Quick inference sanity check

In [ ]:
import matplotlib.pyplot as plt

model.eval()
images, labels = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    logits = tta_forward(model, images)
    preds = logits.argmax(1).cpu().numpy()

images = images.cpu()
labels = labels.cpu().numpy()

mean_t = torch.tensor(mean).view(3,1,1)
std_t = torch.tensor(std).view(3,1,1)

plt.figure(figsize=(16, 8))
num_show = min(8, len(images))
for i in range(num_show):
    img = images[i] * std_t + mean_t
    img = img.permute(1,2,0).clamp(0,1)
    plt.subplot(2, 4, i+1)
    plt.imshow(img)
    correct = "✅" if preds[i] == labels[i] else "❌"
    plt.title(f"{correct} P: {class_names[preds[i]]}\nT: {class_names[labels[i]]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 10) Outputs

In [ ]:
print("📦 Saved files:")
for p in [
    MODEL_TAG,
    "training_history.json",
    "class_to_idx.json",
    "confusion_matrix.png",
    "training_history.png",
]:
    if os.path.exists(p):
        print("-", p)